# Session 23: Ship it

You will repeat a draw with a seed, audit two rules and a model for fairness by group, save the model with joblib, write the app file and complete a model card. The data is the fictional Nile Microfinance file `loans.csv`. Where a cell says `None  # your code here`, replace it with your answer. The checks at the end of each step say OK or NOT YET.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
# Imports and a small helper so that a check never crashes while your answer is still None.
import os
import numpy as np
import pandas as pd

def safe(f):
    try:
        return f()
    except Exception:
        return float("nan")

---
## Step 1: Repeat it: a seed and the library versions

A random seed makes a random draw repeatable. A colleague who uses the same seed gets the same loans.

**PREDICT first.** You draw 5 loans twice with `random_state=42`, and once with `random_state=43`. Which draws will match?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
loans = pd.read_csv("loans.csv")
s1 = None  # your code here: draw 5 loans with random_state=42 and keep their row numbers (.index.tolist())
s2 = None  # your code here: the same draw again
s3 = None  # your code here: the draw with random_state=43
print(s1, s2, s3)
print("numpy", np.__version__, "pandas", pd.__version__)
check("seed 42, first draw", safe(lambda: s1), [2215, 2582, 1662, 3027, 4343], 0)
check("seed 42, second draw", safe(lambda: s2), [2215, 2582, 1662, 3027, 4343], 0)
check("seed 43 draw", safe(lambda: s3), [5762, 1863, 1717, 2500, 2475], 0)

**READ the output.** Which two lists are identical? Which library versions did your session print? Write them next to your results.

> My answer: 

**CHANGE one thing.** Change the seed in the first draw to 7. Do the two seed-42 draws still match each other?

> What I saw: 

---
## Step 2: Default rates by group

Before any rule, look at the default rate for each gender and district. First tidy the district spellings.

**PREDICT first.** Women and men: will the default rates differ by more than 5 percentage points?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
loans["district"] = loans.district.str.strip().str.title()   # given: the cleaning from Session 8
by_gender = None  # your code here: the mean of default for each gender (groupby)
by_district = None  # your code here: the mean of default for each district
print(by_gender)
print(by_district)
check("default rate, women", safe(lambda: by_gender["F"]), 0.1214, 0.002)
check("default rate, Masaka", safe(lambda: by_district["Masaka"]), 0.1482, 0.002)

**READ the output.** Which district has the highest default rate? How many districts are there after tidying?

> My answer: 

**CHANGE one thing.** Skip the tidy line and run again. How many districts do you get, and why is that wrong?

> What I saw: 

**Split the loans once.** From here on, everything that is learned from the data (the occupation default rates, the model) is learned from the 6,000 training loans, and every audit is run on the 2,000 held-out (test) loans. This is the same stratified split as Sessions 14 to 17, so you get the same 2,000 loans. A blank income stays blank for now: the model Pipeline fills it later, using the training loans only.

In [ ]:
from sklearn.model_selection import train_test_split

loans["instalment_to_income"] = loans.loan_amount_ugx * 1.15 / loans.term_months / loans.monthly_income_ugx   # the 1.15 is an ILLUSTRATIVE charge factor, not in the data
features = ["monthly_income_ugx", "loan_amount_ugx", "term_months", "previous_late_payments"]   # a deliberately simplified four-variable model
X = loans[features]
y = loans.default
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
train = loans.loc[X_train.index]   # the 6,000 training loans
test = loans.loc[X_test.index]   # the 2,000 held-out loans
print(len(train), len(test), "defaults in the test loans:", y_test.sum())
check("training loans", len(train), 6000, 0)
check("held-out loans", len(test), 2000, 0)
check("defaults in the held-out loans", y_test.sum(), 251, 0)
check("same held-out loans as Sessions 14 to 17 (sum of the row numbers)", sum(X_test.index), 8081998, 0)

---
## Step 3: Audit rule A: the instalment rule

Rule A flags a loan as high risk when instalment_to_income is above 0.40 (about 35 percent of income, before the illustrative 15 percent charge). We audit it on the held-out loans that have an income. Approval rate is the share of loans not flagged. Disparate impact is the lower approval rate divided by the higher.

**PREDICT first.** The rule never uses gender. Will women and men have similar approval rates?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
known = test.dropna(subset=["monthly_income_ugx"]).copy()   # held-out loans with an income
known["flag"] = (known.instalment_to_income > 0.40).astype(int)
approval_a = None  # your code here: one minus the mean flag for each gender
di_a = None  # your code here: the lower approval rate divided by the higher
print(approval_a)
print(di_a)
check("rule A approval, women", safe(lambda: approval_a["F"]), 0.6509, 0.002)
check("rule A disparate impact", safe(lambda: di_a), 0.9830, 0.003)

**READ the output.** Is the disparate impact close to 1? What does that tell you about the two groups?

> My answer: 

**CHANGE one thing.** Change 0.40 to 0.30. What happens to the two approval rates and to the disparate impact?

> What I saw: 

---
## Step 4: Audit rule B: the occupation rule

Rule B gives each loan the default rate of its occupation, learned from the training loans, and flags it above 12 percent. It never sees gender either.

**PREDICT first.** Women are 77 percent of tailors and 73 percent of salon owners. Will rule B approve women and men equally?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
occ_rate = train.groupby("occupation").default.mean()   # learned from the training loans only
known["flag2"] = (known.occupation.map(occ_rate) > 0.12).astype(int)
approval_b = None  # your code here: one minus the mean of flag2 for each gender
di_b = None  # your code here: the lower approval rate divided by the higher
print(approval_b)
print(di_b)
check("rule B approval, women", safe(lambda: approval_b["F"]), 0.7026, 0.002)
check("rule B disparate impact", safe(lambda: di_b), 0.7453, 0.003)

**READ the output.** Is the disparate impact below the 0.8 rule of thumb (a screening flag, not a legal test)? Which column acts as the proxy variable?

> My answer: 

**CHANGE one thing.** Print `known.groupby("occupation").flag2.mean()`. Which occupations does the rule flag?

> What I saw: 

---
## Step 5: Error rates: who does the rule miss?

A false negative is a loan that defaulted but was not flagged. The false negative rate is the share of defaults the rule missed.

**PREDICT first.** Under rule B, will the share of missed defaults be the same for women and men?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
defaults = known[known.default == 1]   # only the loans that really defaulted
fnr = None  # your code here: one minus the mean of flag2 among the defaults, for each gender
print(fnr)
check("missed defaults, women", safe(lambda: fnr["F"]), 0.6567, 0.002)
check("missed defaults, men", safe(lambda: fnr["M"]), 0.4231, 0.002)

**READ the output.** Which group has more defaults that the rule misses? Who is harmed: the lender, the borrower, or both?

> My answer: 

**CHANGE one thing.** Repeat the calculation for rule A (use `flag`). Are the two groups treated more alike?

> What I saw: 

---
## Step 6: Fit, save and load the model

A scikit-learn pipeline chains an imputer, a scaler and a logistic regression (default C = 1.0, L2 penalty). Fit it on the 6,000 training loans only, so the median and the scaling come from the training loans. joblib writes the trained pipeline to a file, saved in your Colab folder. Load files only from sources you trust: joblib uses pickle underneath.

**PREDICT first.** After you save and load the model, will its predictions differ from the original in any decimal place?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib

from sklearn.metrics import roc_auc_score

model = None  # your code here: a pipeline of an imputer (median), a scaler and a logistic regression
# your code here: fit the model on the TRAINING loans, X_train and y_train
# your code here: save the model to loan_model.joblib with joblib.dump
loaded = None  # your code here: load the file back with joblib.load
p_first = None  # your code here: the probability of default (column 1 of predict_proba) for the first loan
auc_test = None  # your code here: roc_auc_score of y_test and the predicted probability of default (column 1) for X_test: the held-out AUC
print(p_first)
print("held-out AUC:", auc_test)
diff = safe(lambda: np.abs(loaded.predict_proba(X_test) - model.predict_proba(X_test)).max())
print("largest difference after reload:", diff)
check("default probability, first loan", safe(lambda: p_first), 0.2206, 0.01)
check("held-out AUC (simplified four-variable model)", safe(lambda: auc_test), 0.7269, 0.01)
check("reloaded model gives identical predictions", diff, 0, 1e-9)

**READ the output.** What is the probability for the first loan, and what is the held-out AUC? Why do we compare the loaded model with the original? Your model card in Step 9 will quote the held-out AUC.

> My answer: 

**CHANGE one thing.** Change `strategy="median"` to `"mean"` in a copy of the pipeline. Does the first-loan probability move a lot?

> What I saw: 

---
## Step 7: Audit the model by group

Flag a loan as high risk when the model gives a default probability of 0.15 or more, then compute the approval rate for each gender on the 2,000 held-out loans.

**PREDICT first.** The model never sees gender and does not use occupation. Will its disparate impact be closer to rule A or rule B?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
flag_model = None  # your code here: True where the default probability for a held-out loan is 0.15 or more
approval_m = None  # your code here: one minus the mean flag for each gender
di_m = None  # your code here: the lower approval rate divided by the higher
print(approval_m)
print(di_m)
check("model flag rate on the held-out loans", safe(lambda: np.mean(flag_model)), 0.2800, 0.02)
check("model approval, women", safe(lambda: approval_m["F"]), 0.7231, 0.01)
check("model disparate impact", safe(lambda: di_m), 0.9895, 0.02)

**READ the output.** Is the disparate impact near 1? Would you release this model on this evidence alone? What else would you check?

> My answer: 

**CHANGE one thing.** Change the threshold to 0.20. What happens to the approval rates?

> What I saw: 

---
## Step 8: Write app.py and requirements.txt

These two files, plus loan_model.joblib, are saved in your Colab folder and are what Streamlit Community Cloud needs. You do not run the app here. You deploy it later from your own GitHub repository.

**PREDICT first.** How many lines does the app file have? Which line loads the model?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
app_code = '''import joblib
import pandas as pd
import streamlit as st

model = joblib.load('loan_model.joblib')

st.title('Loan risk checker')
st.caption('A person decides.')

income = st.number_input('Income (UGX)',
                         value=600000)
amount = st.number_input('Loan (UGX)',
                         value=1500000)
months = st.slider('Months', 3, 12, 6)
late = st.slider('Late payments', 0, 6, 0)

row = pd.DataFrame({
  'monthly_income_ugx': [income],
  'loan_amount_ugx': [amount],
  'term_months': [months],
  'previous_late_payments': [late]})

if st.button('Show probability'):
  p = model.predict_proba(row)[0, 1]
  st.metric('Default chance', f'{p:.0%}')
'''
# your code here: write app_code to a file called app.py
req = None  # your code here: a list with one line per library: numpy, pandas, scikit-learn, joblib, streamlit, as name==version (name only if not installed)
open("requirements.txt", "w").write("\n".join(req or []) + "\n")
app_text = open("app.py").read() if os.path.exists("app.py") else ""
try:
    compile(app_text, "app.py", "exec")
    compiles = 1 if app_text else 0
except SyntaxError:
    compiles = 0
print(len(app_text.splitlines()), "lines in app.py;", len(open("requirements.txt").read().split()), "libraries in requirements.txt")
check("app.py has 25 lines", len(app_text.splitlines()), 25, 0)
check("app.py is valid Python", compiles, 1, 0)
check("app.py calls predict_proba", int("predict_proba" in app_text), 1, 0)
check("requirements.txt lists 5 libraries", len(open("requirements.txt").read().split()), 5, 0)

**READ the output.** Which line of app.py loads the model and which shows the result? Why must requirements.txt list streamlit?

> My answer: 

**CHANGE one thing.** Change the caption in app_code to your own sentence, write the file again and read it back. If an app host cannot install a pinned version, remove the version number of that library from requirements.txt and test again.

> What I saw: 

---
## Step 9: The model card and the data protection checklist

Write what the model is for, what it learned from, how well it works, where it fails and who reviews it. Then say what you do for each of the seven principles in the Data Protection and Privacy Act, 2019 (a teaching summary, not legal advice).

**PREDICT first.** Which section of the card would a manager read first? Which principle is hardest for your own project?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
model_card = {
    "purpose": None,   # your code here: one sentence
    "data": None,   # your code here: source, years, what you removed and why
    "metrics": None,   # your code here: the held-out AUC from Step 6 and the share of held-out loans flagged
    "limits": None,   # your code here: at least one limit
    "fairness_and_review": None,   # your code here: the approval rates and disparate impact from step 7 (held-out loans), and who reviews
}
dp_checklist = {
    "Accountability": None,   # your code here: what you do, or still need to do
    "Fair and lawful processing": None,
    "Adequate and relevant data": None,
    "Kept only as long as needed": None,
    "Quality of information": None,
    "Openness and participation": None,
    "Safety and security": None,
}
for k, v in model_card.items():
    print(k, ":", v)
check("model card has 5 sections", sum(bool(v) for v in model_card.values()), 5, 0)
check("checklist has 7 principles", sum(bool(v) for v in dp_checklist.values()), 7, 0)
show_checks()

**READ the output.** Does your card state a limit? A limit stated openly builds trust. A hidden one costs it.

> My answer: 

**CHANGE one thing.** Add a sixth key to model_card called "next_review" with a date, and print the card again.

> What I saw: 

## What you keep

`loan_model.joblib`, `app.py` and `requirements.txt` are saved in your Colab folder (the Files panel on the left). Download them and this notebook into one folder. Create a free GitHub account, make a new public repository, and upload the folder after checking that it holds no personal data. Then, if you wish, deploy the app from Streamlit Community Cloud. At the time of writing it is free for public repositories. Check the current limits.